# Finalne Porównanie Modeli: EKG vs HR Detection

Ten notebook zestawia dwa podejścia do analizy sygnałów mechanokardiograficznych (SCG/GCG):
1. **Model Rekonstrukcji (Regresja)**: Odtwarzanie pełnej fali EKG.
2. **Model Detekcji (Klasyfikacja)**: Bezpośrednie przewidywanie pików uderzeń serca.

Ewaluacja przeprowadzona na rekordzie `CP-01` (Zenodo).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Importy z lokalnego repozytorium
from data_loader import DataLoader as ECGDataLoader
from preprocessing import Preprocessor
from models.model import ECGReconstructionModel
from models.model_hr import HRVBeatDetectionModel
from utils_peaks import extract_r_peaks, refine_peak_parabolic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

## 1. Ładowanie Modeli i Danych

In [ ]:
# Ścieżki do modeli
ECG_MODEL_PATH = 'global_best_ecg_model.pth'
HR_MODEL_PATH = 'models/global_best_hr_model.pth'

# Inicjalizacja modeli
model_ecg = ECGReconstructionModel().to(device)

# Bezpieczne ładowanie wag (obsługa różnej długości pos_embedding)
try:
    state_dict_ecg = torch.load(ECG_MODEL_PATH, map_location=device)
    model_dict = model_ecg.state_dict()
    for k, v in state_dict_ecg.items():
        if k == 'pos_embedding' and v.shape != model_dict[k].shape:
            # Jeśli wagi mają inny wymiar (np. 1000) a model ma 4000, kopiujemy tylko fragment
            model_dict[k][:, :v.size(1), :] = v
        else:
            model_dict[k] = v
    model_ecg.load_state_dict(model_dict)
    print(f" -> Załadowano wagi modelu ECG z: {ECG_MODEL_PATH}")
except Exception as e:
    print(f"Błąd ładowania modelu ECG: {e}")

model_ecg.eval()

model_hr = HRVBeatDetectionModel(input_dim=1, hidden_dim=64, num_layers=2).to(device)
model_hr.load_state_dict(torch.load(HR_MODEL_PATH, map_location=device))
model_hr.eval()

# Wczytanie danych (CP-01)
loader = ECGDataLoader()
pre = Preprocessor(fs=256)
record_name = 'CP-01'
df = loader.load_zenodo(record=record_name, format=True)
results = pre.process_pipeline(df)

print(f"Załadowano rekord {record_name} i wagi modeli globalnych (z poprawką na długość okna).")

In [ ]:
# Przygotowanie sygnałów
scg_full = results['scg_kaisti']
pcg_full = results['gcg_kaisti']
ecg_full = results['ecg_kaisti']
fs = 256
clean_mask = results['clean_mask']
epoch_sec = 10
n_samples_epoch = int(epoch_sec * fs)

# Wybór pierwszego czystego segmentu 10s
clean_indices = np.where(clean_mask)[0]
start_sec = clean_indices[0] * epoch_sec if len(clean_indices) > 0 else 0
visualize_start = int(start_sec * fs)
visualize_end = visualize_start + 10 * fs

print(f"Wybrano segment do porównania: {start_sec}s - {start_sec+10}s")

## 2. Inferencja i Detekcja Pików

In [ ]:
# 1. Przygotowanie okna do modeli
scg_win = scg_full[visualize_start:visualize_end]
pcg_win = pcg_full[visualize_start:visualize_end]

# Normalizacja okienkowa (jak w treningu globalnym)
scg_win_norm = (scg_win - np.mean(scg_win)) / (np.std(scg_win) + 1e-9)
pcg_win_norm = (pcg_win - np.mean(pcg_win)) / (np.std(pcg_win) + 1e-9)

t_scg = torch.tensor(scg_win_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
t_pcg = torch.tensor(pcg_win_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)

with torch.no_grad():
    # Model 1: Rekonstrukcja EKG
    ecg_recon = model_ecg(t_pcg, t_scg).cpu().squeeze().numpy()
    
    # Model 2: Maska HR
    hr_mask = model_hr(t_pcg, t_scg).cpu().squeeze().numpy()

# 2. Detekcja Pików (Ground Truth)
gt_peaks_global = extract_r_peaks(ecg_full, fs=fs)
gt_peaks_win = gt_peaks_global[(gt_peaks_global >= visualize_start) & (gt_peaks_global < visualize_end)] - visualize_start

# 3. Detekcja Pików (z Rekonstrukcji EKG)
recon_peaks_win = extract_r_peaks(ecg_recon, fs=fs)

# 4. Detekcja Pików (z Maski HR + Parabolic Interpolation)
max_p = np.max(hr_mask)
thresh = 0.5 * max_p if max_p > 0.1 else 0.3
hr_peaks_discrete, _ = find_peaks(hr_mask, height=thresh, distance=int(0.3 * fs))
hr_peaks_win = [refine_peak_parabolic(hr_mask, p) for p in hr_peaks_discrete]

print(f"Liczba wykrytych uderzeń -> GT: {len(gt_peaks_win)}, Recon: {len(recon_peaks_win)}, HR-Mask: {len(hr_peaks_win)}")

## 3. Zestawienie Metryk HRV

In [ ]:
def get_hrv_metrics(peaks, fs=256):
    if len(peaks) < 2: return {'BPM': 0, 'SDNN': 0, 'RMSSD': 0}
    rr_ms = np.diff(peaks) / fs * 1000.0
    valid_rr = rr_ms[(rr_ms > 300) & (rr_ms < 1500)]
    if len(valid_rr) < 2: return {'BPM': 0, 'SDNN': 0, 'RMSSD': 0}
    return {
        'BPM': np.round(60000.0 / np.mean(valid_rr), 1),
        'SDNN': np.round(np.std(valid_rr, ddof=1), 2),
        'RMSSD': np.round(np.sqrt(np.mean(np.diff(valid_rr)**2)), 2)
    }

metrics = {
    'Source': ['Ground Truth', 'ECG Reconstruction', 'HR Beat Mask'],
    'BPM': [get_hrv_metrics(gt_peaks_win)['BPM'], get_hrv_metrics(recon_peaks_win)['BPM'], get_hrv_metrics(hr_peaks_win)['BPM']],
    'SDNN': [get_hrv_metrics(gt_peaks_win)['SDNN'], get_hrv_metrics(recon_peaks_win)['SDNN'], get_hrv_metrics(hr_peaks_win)['SDNN']],
    'RMSSD': [get_hrv_metrics(gt_peaks_win)['RMSSD'], get_hrv_metrics(recon_peaks_win)['RMSSD'], get_hrv_metrics(hr_peaks_win)['RMSSD']]
}

df_metrics = pd.DataFrame(metrics)
display(df_metrics)

## 4. Wizualizacja (Visual Battle)

In [ ]:
t = np.arange(len(scg_win)) / fs

plt.figure(figsize=(15, 12))

# Subplot 1: Oryginalne EKG vs Rekonstrukcja
plt.subplot(3, 1, 1)
plt.plot(t, ecg_full[visualize_start:visualize_end], label='EKG (Reference)', color='red', alpha=0.5)
plt.plot(t, ecg_recon, label='EKG (Reconstructed)', color='blue', linestyle='--')
plt.title('Porównanie: EKG Oryginalne vs Rekonstrukcja Modelu 1')
plt.legend()

# Subplot 2: Sygnał SCG + Maska HR
plt.subplot(3, 1, 2)
plt.plot(t, scg_win_norm, label='SCG (Normalized)', color='gray', alpha=0.3)
plt.plot(t, hr_mask, label='HR Beat Probability Mask', color='green', linewidth=2)
plt.title('Maska Prawdopodobieństwa Modelu 2 na tle sygnału SCG')
plt.legend()

# Subplot 3: Synchronizacja czasowa wszystkich metod
plt.subplot(3, 1, 3)
plt.vlines(gt_peaks_win / fs, 0.8, 1, colors='r', label='GT Peaks', linewidth=2)
plt.vlines(recon_peaks_win / fs, 0.4, 0.6, colors='b', label='Recon Peaks', linestyles='--')
plt.vlines(np.array(hr_peaks_win) / fs, 0, 0.2, colors='g', label='HR-Mask Peaks')
plt.yticks([0.1, 0.5, 0.9], ['HR Mask', 'Recon EKG', 'GT EKG'])
plt.title('Synchronizacja Czasowa Wykrytych Pików')
plt.xlabel('Czas (s)')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Podsumowanie Eksperckie: Analiza Stabilności i Jitteru

### Wnioski Techniczne:
1.  **Jitter w RMSSD**: Model detekcji bezpośredniej (`HRVBeatDetectionModel`), mimo zastosowania interpolacji parabolicznej, wciąż wykazuje większą tendencję do mikrosekundowych wahań (jitteru) niż model rekonstrukcji. Wynika to z faktu, że rekonstrukcja fali EKG (regresja) "uśrednia" kształt cyklu, co działa dodatkowo jako filtr stabilizujący dla algorytmu Pan-Tompkins.
2.  **Rola Fuzji SCG+GCG**: 
    *   Fuzja dwóch modalności mechanicznych była kluczowa dla stabilności izowolumetrycznej fazy skurczu. 
    *   W nagraniach z bazy Zenodo (takich jak `CP-01`), sygnał SCG często zawiera artefakty oddechowe, które GCG (żyroskop) skutecznie kompensuje, zapobiegając "gubieniu" bić serca w masce prawdopodobieństwa.
3.  **Wybór Modelu**: 
    *   Do celów **medycznej diagnostyki morfologii EKG** (np. analiza odcinka ST) lepszy jest `ECGReconstructionModel`. 
    *   Do celów **długotrwałego monitoringu HR** w trudnych warunkach ruchowych, `HRVBeatDetectionModel` (klasyfikacja) jest bardziej odporny na całkowite zgubienie rytmu dzięki funkcji straty DiceBCE.
